Document Parsers and chunking using LlamaIndex

https://developers.llamaindex.ai/python/framework/module_guides/loading/node_parsers/modules/

refer above for more for different kinds of node parsers

In [ ]:
#Importing libraries

from llama_index.core import VectorStoreIndex, Settings, Document, SimpleDirectoryReader
from llama_index.core.node_parser import (
    SentenceSplitter, TokenTextSplitter, SemanticSplitterNodeParser
)

from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding

from llama_index.core.extractors import TitleExtractor, SummaryExtractor

from llama_index.vector_stores import qdrant, chroma

import os
from pathlib import Path
from dotenv import load_dotenv
from datetime import datetime
import warnings
warnings.filterwarnings("ignore")




True

In [8]:

load_dotenv()


True

In [5]:
main_dir = Path('./data')

research_dir = main_dir/'research_papers'

if Path(research_dir).exists():
    papers = list(research_dir.glob("*.pdf"))
    print(f"No of research papers found {len(papers)}")

    for paper in papers:
        print(f"Filename : {paper.name}")


No of research papers found 3
Filename : attention_paper.pdf
Filename : docling_paper.pdf
Filename : scan_notes.pdf


In [6]:
!uv pip install llama_parse

Resolved 64 packages in 4.65s
Prepared 3 packages in 2.79s
Installed 3 packages in 749ms
 + llama-cloud==0.1.46
 + llama-cloud-services==0.6.94
 + llama-parse==0.6.94


In [12]:
from llama_parse import LlamaParse
LLAMA_CLOUD_API_KEY = os.getenv("LLAMA_CLOUD_API_KEY")
pdfparse  = LlamaParse(
    api_key=LLAMA_CLOUD_API_KEY,
    show_progress = True,
    result_type="markdown"
)

filename = research_dir/"docling_paper.pdf"
extra_info = {'filename':filename}
if filename.exists():
   documents = pdfparse.load_data(filename,extra_info=extra_info)

Started parsing the file under job_id ffe21c6d-f62b-4fe5-83b1-911ff1fd593e
.

In [15]:
print(f"No of documents created from the file {filename} is {len(documents)}")

for i,doc in enumerate(documents,1):
    if i==1:
        print(doc.metadata)
    

No of documents created from the file data\research_papers\docling_paper.pdf is 9
{'filename': WindowsPath('data/research_papers/docling_paper.pdf')}


## Chunking Strategies

In [25]:
from llama_index.core.node_parser import SentenceSplitter

chunker = SentenceSplitter(
    chunk_size=1024,
    chunk_overlap=200,
    separator='.'
)

sentence_chunks=chunker.get_nodes_from_documents(documents=documents)

print(f"The number of {len(sentence_chunks)}")

The number of 13


In [26]:
print(sentence_chunks[0].metadata)

{'filename': WindowsPath('data/research_papers/docling_paper.pdf')}


In [27]:
print(f"Avg len : {sum(len(c.text) for c in sentence_chunks)/(len(sentence_chunks))}")

Avg len : 2812.846153846154


In [19]:
from llama_index.core.node_parser import TokenTextSplitter

chunker = TokenTextSplitter(chunk_size=1024,
                            chunk_overlap=200,
                            separator=' ',
                            )

token_chunks = chunker.get_nodes_from_documents(documents=documents,show_progress=True,
                                                )

print(f"The number of chunks from the token splitter {len(token_chunks)}")

print(token_chunks[0])

Parsing nodes: 100%|██████████| 9/9 [00:00<00:00, 30.06it/s]

The number of chunks from the token splitter 15
Node ID: aa5c8d5b-ed28-4156-ac42-503f7e96bf74
Text: arXiv:2408.09869v5 [cs.CL] 9 Dec 2024  # Docling Technical
Report  Version 1.0  Christoph Auer, Maksym Lysak, Ahmed Nassar,
Michele Dolfi, Nikolaos Livathinos, Panos Vagenas, Cesar Berrospi
Ramis, Matteo Omenetti, Fabian Lindlbauer, Kasper Dinkla, Lokesh
Mishra, Yusik Kim, Shubham Gupta, Rafael Teixeira de Lima, Valery
Weber, Lucas Morin, Ingmar...


In [22]:
print(f"Avg len : {sum(len(c.text) for c in token_chunks)/(len(token_chunks))}")

Avg len : 2486.0


In [ ]:
Settings.embed_model = OpenAIEmbedding(model = 'text-embedding-3-small',
                                       dimension=1536)

In [43]:
from llama_index.core.node_parser import SemanticSplitterNodeParser

chunker = SemanticSplitterNodeParser(embed_model=Settings.embed_model,
                                     buffer_size=1,
                                     breakpoint_percentile_threshold=95)

semantic_chunks = chunker.get_nodes_from_documents(documents=documents)

print(f"No of the chunks {len(semantic_chunks)}")

No of the chunks 24


In [44]:
print(semantic_chunks[0])

Node ID: fc780c2f-4896-476f-ab0e-0fb590da5f7e
Text: arXiv:2408.09869v5 [cs.CL] 9 Dec 2024  # Docling Technical
Report  Version 1.0  Christoph Auer, Maksym Lysak, Ahmed Nassar,
Michele Dolfi, Nikolaos Livathinos, Panos Vagenas, Cesar Berrospi
Ramis, Matteo Omenetti, Fabian Lindlbauer, Kasper Dinkla, Lokesh
Mishra, Yusik Kim, Shubham Gupta, Rafael Teixeira de Lima, Valery
Weber, Lucas Morin, Ingmar...


In [45]:
print(f"Avg len : {sum(len(c.text) for c in semantic_chunks)/(len(semantic_chunks))}")

Avg len : 1408.375


In [46]:
for i,c in enumerate(semantic_chunks,1):
    c.metadata['index_id']=i
    c.metadata['strategy']='semantic'
    c_text = c.text.lower() 
    c.metadata['has_abstract']= 'abstract' in c_text

print(semantic_chunks[0].metadata)
    

{'filename': WindowsPath('data/research_papers/docling_paper.pdf'), 'index_id': 1, 'strategy': 'semantic', 'has_abstract': True}


In [47]:
open_ai_key = os.getenv('OPENAI_API_KEY')

In [48]:
Settings.llm = OpenAI(model = 'gpt-4o-mini',api_key=open_ai_key)

In [52]:
##Extracing the summary and title from the text

from llama_index.core.extractors import TitleExtractor, SummaryExtractor

title_ex = TitleExtractor(llm=Settings.llm,
                       nodes=5)
title_chunks = title_ex.process_nodes(semantic_chunks[:2])
print(title_chunks[0].metadata['document_title'])

for i, c in enumerate(semantic_chunks,1):
    c.metadata['title'] = title_chunks[0].metadata["document_title"]

#print(semantic_chunks[0].metadata)

  0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:02<00:00,  2.56s/it]

"Docling: An Open-Source AI-Powered PDF Document Conversion Tool for Enhanced Layout and Table Recognition"


In [54]:
from llama_index.core.extractors import SummaryExtractor

summary_extractor = SummaryExtractor(llm=Settings.llm,api_key=open_ai_key)

summary_chunks = summary_extractor.process_nodes(semantic_chunks[1:6])

print(summary_chunks[0].metadata)

  0%|          | 0/5 [00:00<?, ?it/s]

100%|██████████| 5/5 [00:07<00:00,  1.59s/it]

{'filename': WindowsPath('data/research_papers/docling_paper.pdf'), 'index_id': 2, 'strategy': 'semantic', 'has_abstract': False, 'document_title': '"Docling: An Open-Source AI-Powered PDF Document Conversion Tool for Enhanced Layout and Table Recognition"', 'title': '"Docling: An Open-Source AI-Powered PDF Document Conversion Tool for Enhanced Layout and Table Recognition"', 'section_summary': 'The section introduces "Docling," an open-source AI-powered tool aimed at converting PDF documents into machine-processable formats. Key topics discussed include:\n\n1. **Challenges of PDF Conversion**: The difficulties arise from the variability of PDF formats, weak standardization, and their design focus on printing rather than structural features and metadata.\n\n2. **Advancements in Document Understanding**: The rise of large language models (LLMs) and techniques like retrieval-augmented generation (RAG) have made it increasingly important to extract content from PDFs.\n\n3. **Market Landsc

In [56]:
print(summary_chunks[0].metadata["section_summary"])

The section introduces "Docling," an open-source AI-powered tool aimed at converting PDF documents into machine-processable formats. Key topics discussed include:

1. **Challenges of PDF Conversion**: The difficulties arise from the variability of PDF formats, weak standardization, and their design focus on printing rather than structural features and metadata.

2. **Advancements in Document Understanding**: The rise of large language models (LLMs) and techniques like retrieval-augmented generation (RAG) have made it increasingly important to extract content from PDFs.

3. **Market Landscape**: The document highlights the existence of various commercial and cloud-based document understanding solutions, noting a lack of open-source alternatives that match the features and quality of proprietary tools.

4. **Docling's Features**: Docling is presented as an efficient solution that employs specialized AI models for layout analysis and table structure recognition. It is a self-contained Pyt

In [ ]:
print(summary_chunks[0].metadata["section_summary"])

"Docling: An Open-Source AI-Powered PDF Document Conversion Tool for Enhanced Layout and Table Recognition"


In [63]:
print(semantic_chunks[7].metadata["title"])

"Docling: An Open-Source AI-Powered PDF Document Conversion Tool for Enhanced Layout and Table Recognition"


In [67]:
print(semantic_chunks[1].metadata["document_title"])

"Docling: An Open-Source AI-Powered PDF Document Conversion Tool for Enhanced Layout and Table Recognition"


### Check Node Relationship

In [72]:
from llama_index.core.schema import TextNode, NodeRelationship, RelatedNodeInfo

In [74]:
for i, node in enumerate(semantic_chunks,1):
    print(f"Node #: {i}")
    print(f"node id : {node.node_id}")
    print(f"Relationships : {list(node.relationships.keys())}")


    if NodeRelationship.SOURCE in node.relationships:
        source_info = node.relationships[NodeRelationship.SOURCE]
        print(f"Source info : {source_info}")

    if NodeRelationship.PREVIOUS in node.relationships:
        print("Has Previous Node")
    if NodeRelationship.NEXT in node.relationships:
        print("Has Next Node")


Node #: 1
node id : fc780c2f-4896-476f-ab0e-0fb590da5f7e
Relationships : [<NodeRelationship.SOURCE: '1'>, <NodeRelationship.NEXT: '3'>]
Source info : node_id='c7268da0-91fb-4d2b-bd38-0a54a8706d11' node_type=<ObjectType.DOCUMENT: '4'> metadata={'filename': WindowsPath('data/research_papers/docling_paper.pdf')} hash='3174205c133bbba8c0973217eb9008e5137246f38cc068e089009f54443f8c7f'
Has Next Node
Node #: 2
node id : 0642c6b1-c5de-4f05-b652-90904cfb906b
Relationships : [<NodeRelationship.SOURCE: '1'>, <NodeRelationship.PREVIOUS: '2'>]
Source info : node_id='c7268da0-91fb-4d2b-bd38-0a54a8706d11' node_type=<ObjectType.DOCUMENT: '4'> metadata={'filename': WindowsPath('data/research_papers/docling_paper.pdf')} hash='3174205c133bbba8c0973217eb9008e5137246f38cc068e089009f54443f8c7f'
Has Previous Node
Node #: 3
node id : 42223c80-38ab-4b4d-a363-b20186f8e7c1
Relationships : [<NodeRelationship.SOURCE: '1'>, <NodeRelationship.NEXT: '3'>]
Source info : node_id='8b7a4968-da2d-4ac8-b0db-4755a2c9c2e3' n

In [78]:
summary_node = TextNode(
    text="Summary of the content",
    metadata={'type':'summary', 'level':0}
)

for node in semantic_chunks[:5]:
    node.relationships[NodeRelationship.PARENT]=RelatedNodeInfo(node_id=summary_node.node_id)
    node.metadata['level']=2

print(f"No. of Nodes that has parent node: {len([n for n in semantic_chunks if NodeRelationship.PARENT in n.relationships])}")



No. of Nodes that has parent node: 5


## Ingestion Pipeline

In [82]:
from llama_index.core.ingestion import IngestionPipeline

pipeline = IngestionPipeline(transformations=[SentenceSplitter(chunk_size=1024, chunk_overlap=200),
                                              Settings.embed_model])

print("Starting the Pipeline")
nodes = pipeline.run(documents=documents, show_progress=True)

print(f"No of nodes returned : {len(nodes)}")

Starting the Pipeline


Applying transformations: 100%|██████████| 2/2 [00:01<00:00,  1.36it/s]

No of nodes returned : 13


In [85]:
print(nodes[1].text)

# Here is what Docling delivers today:

- Converts PDF documents to JSON or Markdown format, stable and lightning fast
- Understands detailed page layout, reading order, locates figures and recovers table structures
- Extracts metadata from the document, such as title, authors, references and language
- Optionally applies OCR, e.g. for scanned PDFs
- Can be configured to be optimal for batch-mode (i.e high throughput, low time-to-solution) or interactive mode (compromise on efficiency, low time-to-solution)
- Can leverage different accelerators (GPU, MPS, etc).

# 2 Getting Started

To use Docling, you can simply install the docling package from PyPI. Documentation and examples are available in our GitHub repository at github.com/DS4SD/docling. All required model assets1 are downloaded to a local huggingface datasets cache on first use, unless you choose to pre-install the model assets in advance.

Docling provides an easy code interface to convert PDF documents from file system, URLs 

In [86]:
index = VectorStoreIndex(nodes=nodes)

query_engine = index.as_query_engine(similarity_top_k = 4,
                                     response_mode='compact'
                                     )
q1 = "who introduced docling"

ans1 = query_engine.query(q1)

print(ans1)

Docling was introduced by a team from the AI4K Group at IBM Research, which includes Christoph Auer, Maksym Lysak, Ahmed Nassar, Michele Dolfi, Nikolaos Livathinos, Panos Vagenas, Cesar Berrospi Ramis, Matteo Omenetti, Fabian Lindlbauer, Kasper Dinkla, Lokesh Mishra, Yusik Kim, Shubham Gupta, Rafael Teixeira de Lima, Valery Weber, Lucas Morin, Ingmar Meijer, Viktor Kuropiatnyk, and Peter W. J. Staar.
